Goal: Compare all-at-once prompting vs. one-at-a-time prompting to see whcih gives more accurate and 
reliable percentage predictions from LLM

all-at-once prompt template:

You are a helpful prediction assistant.

A person has the following demographic information:
- [demographic_info] (e.g., Female, 18-24, American Indian, Low income)
- Lives in a [loc_desc] (e.g., Small town or Large city)
- Located in [state_name]

Based on this information, please answer the following survey question:

Question: [user_question]

Here are the answer choices:
- [choice_1]
- [choice_2]
- ...

Please estimate the probability that this person would choose each option.

Return your answer in this exact format (on separate lines, percentages only, no explanation):

[choice_1]: xx%  
[choice_2]: xx%  
...







In [6]:
import os
import pandas as pd
from openai import AzureOpenAI

class PoliticalLLM:
    def __init__(self, vm_file, candidates, api_key, endpoint, deployment_name):
        self.vm = pd.read_csv(vm_file)
        self.candidates = candidates
        self.api_key = api_key
        self.endpoint = endpoint
        self.deployment_name = deployment_name

        # Initialize OpenAI client
        self.client = AzureOpenAI(
            api_key=self.api_key,
            azure_endpoint=endpoint,
            api_version="2024-12-01-preview"
        )
    
    def load_state(self, state_file):
        """Load a new state CSV file and map coded values to descriptions."""
        self.df = pd.read_csv(state_file)
        self.desc_cols = []

        for column in self.df.columns:
            vm_subset = self.vm[self.vm['var_id'] == column]
            if not vm_subset.empty:
                value_dict = dict(zip(vm_subset['value_id'], vm_subset['var_values']))
                description = vm_subset['description'].iloc[0].upper()
                self.df[description] = self.df[column].map(value_dict)
                self.desc_cols.append(description)
        
        self.df = self.df.dropna(subset=self.desc_cols + ['loc_msa'])
    
    def build_prompt(self, sample_row, state_name):
        demographic_info = ', '.join([str(sample_row[col]) for col in self.desc_cols])
        loc_desc = 'Small town' if sample_row['loc_msa'] == 'S' else 'Large city'

        prompt = f"If the person's info is: {demographic_info}, lives in {loc_desc}, in {state_name}\n"
        prompt += "Here is the list of candidates:\n"
        for c in self.candidates:
            prompt += f"- {c}\n"
        prompt += "Based on this person's info, what is the probability that this person votes for each candidate?\n"
        prompt += "Please give your answer in this clean format (each on a new line, and no other information):\n"
        for c in self.candidates:
            prompt += f"{c}: xx%\n"

        return prompt
 
    def call_LLM(self, prompt):
        response = self.client.chat.completions.create(
            model=self.deployment_name,
            messages=[
                {"role": "system", "content": "You are a prediction assistant for any topics. You MUST use demographic and geographic patterns to estimate voting preferences. Avoid saying 50/50 unless truly uncertain."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    
    def poll(self, state_name):
        totals = {c: 0 for c in self.candidates}  # Initialize totals per candidate

        for idx, row in self.df.iterrows():
            print(f"Processing row {idx+1}/{len(self.df)} in state {state_name}...")

            prompt = self.build_prompt(row, state_name)
            prob = self.call_LLM(prompt)
            print(f"Got prediction: {prob}")

            for line in prob.split('\n'):
                line = line.strip()
                if ':' in line:
                    candidate, perc_str = line.split(':')
                    candidate = candidate.strip()
                    perc_str = perc_str.strip().replace('%', '')

                    if candidate in totals and perc_str.replace('.', '', 1).isdigit():
                        percentage = float(perc_str)
                        totals[candidate] += (percentage / 100)

        return totals


# MAIN PROGRAM (No multiprocessing)
if __name__ == "__main__":
    import time

    vm_file = "variable_mapping.csv"
    api_key = input("Enter your Azure OpenAI API key: ")
    endpoint = input("Enter your Azure endpoint URL: ")
    deployment_name = input("Enter your deployment name: ")
    candidates = input("Enter all candidate names separated by commas: ").split(",")

    start_time = time.time()

    state_file = "test_table.csv"
    state_name = "Alaska"

    # Run polling
    poller = PoliticalLLM(vm_file, candidates, api_key, endpoint, deployment_name)
    poller.load_state(state_file)
    results = poller.poll(state_name)

    # Print winner
    winner = max(results, key=results.get)
    print(f"\nThe winner is {winner} with {results[winner]} votes.")
    print(f"Finished in {time.time() - start_time:.2f} seconds")


Processing row 1/50 in state Alaska...
Got prediction: pepsi: 45%
coca cola: 55%
Processing row 2/50 in state Alaska...
Got prediction: pepsi: 45%
coca cola: 55%
Processing row 3/50 in state Alaska...
Got prediction: pepsi: 60%
coca cola: 40%
Processing row 4/50 in state Alaska...
Got prediction: pepsi: 47%
coca cola: 53%
Processing row 5/50 in state Alaska...
Got prediction: pepsi: 60%
coca cola: 40%
Processing row 6/50 in state Alaska...
Got prediction: pepsi: 45%
coca cola: 55%
Processing row 7/50 in state Alaska...
Got prediction: pepsi: 47%
coca cola: 53%
Processing row 8/50 in state Alaska...
Got prediction: pepsi: 45%
coca cola: 55%
Processing row 9/50 in state Alaska...
Got prediction: pepsi: 65%
coca cola: 35%
Processing row 10/50 in state Alaska...
Got prediction: pepsi: 30%
coca cola: 70%
Processing row 11/50 in state Alaska...
Got prediction: pepsi: 45%
coca cola: 55%
Processing row 12/50 in state Alaska...
Got prediction: pepsi: 45%
coca cola: 55%
Processing row 13/50 in s

Based on the output:
1) each of the prompt, LLM returns 2 predicted percentages
2) each output has well-formed numbers between 0 and 100
3) each set percentages add up to exactly 100%
4) each set of output returned by LLM has the correct format
5) each set uses correct candidate names.

So, error rate = 0%

All-at-once approach takes 61.92 seconds.
---------------------------------------------------------------------------------------------------------


one-at-a-time template:

prompt 1:

A person has the following demographic information:
- [demographic_info] (e.g., Female, 18-24, American Indian, Low income)
- Lives in a [loc_desc] (e.g., Small town or Large city)
- Located in [state_name]

Based on this information, please answer the following survey question:
Survey options: [candidates]

With what probability will this person choose [choice_1]? Return your answer in this exact format (on separate lines, percentages only, no explanation)


prompt 2: 
A person has the following demographic information:
- [demographic_info] (e.g., Female, 18-24, American Indian, Low income)
- Lives in a [loc_desc] (e.g., Small town or Large city)
- Located in [state_name]

Based on this information, please answer the following survey question:
Survey options: [candidates]

With what probability will this person choose [choice_2] ?Return your answer in this exact format (on separate lines, percentages only, no explanation)

In [7]:
import os
import time
import pandas as pd
from openai import AzureOpenAI

class PoliticalLLM:
    def __init__(self, vm_file, candidates, api_key, endpoint, deployment_name):
        self.vm = pd.read_csv(vm_file)
        self.candidates = [c.strip() for c in candidates]  # Clean candidate names
        self.api_key = api_key
        self.endpoint = endpoint
        self.deployment_name = deployment_name

        self.client = AzureOpenAI(
            api_key=self.api_key,
            azure_endpoint=endpoint,
            api_version="2024-12-01-preview"
        )
    
    def load_state(self, state_file):
        self.df = pd.read_csv(state_file)
        self.desc_cols = []

        for column in self.df.columns:
            vm_subset = self.vm[self.vm['var_id'] == column]
            if not vm_subset.empty:
                value_dict = dict(zip(vm_subset['value_id'], vm_subset['var_values']))
                description = vm_subset['description'].iloc[0].upper()
                self.df[description] = self.df[column].map(value_dict)
                self.desc_cols.append(description)

        self.df = self.df.dropna(subset=self.desc_cols + ['loc_msa'])
    
    def build_prompt(self, sample_row, state_name, current_candidate):
        demographic_info = ', '.join([str(sample_row[col]) for col in self.desc_cols])
        loc_desc = 'Small town' if sample_row['loc_msa'] == 'S' else 'Large city'
        candidate_list = ', '.join(self.candidates)

        prompt = (
            "You are a helpful prediction assistant.\n\n"
            f"A person has the following demographic information:\n"
            f"- {demographic_info}\n"
            f"- Lives in a {loc_desc}\n"
            f"- Located in {state_name}\n\n"
            "Based on this information, please answer the following survey question:\n"
            f"Survey options: {candidate_list}\n"
            f"With what probability will this person choose {current_candidate}?\n\n"
            f"Return your answer in this exact format (on a single line, no explanation):\n"
            f"{current_candidate}: xx%"
        )
        return prompt
 
    def call_LLM(self, prompt):
        response = self.client.chat.completions.create(
            model=self.deployment_name,
            messages=[
                {"role": "system", "content": "You are a prediction assistant for any topics. You MUST use demographic and geographic patterns to estimate voting preferences. Avoid saying 50/50 unless truly uncertain."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    
    def poll(self, state_name):
        totals = {c: 0 for c in self.candidates}
        skipped_rows = 0

        for idx, row in self.df.iterrows():
            print(f"\nProcessing row {idx + 1}/{len(self.df)} in state {state_name}...")

            raw_predictions = {}  # Store raw percentages per candidate

            for c in self.candidates:
                prompt = self.build_prompt(row, state_name, c)
                response = self.call_LLM(prompt)
                print(f"{c}: {response}")

                # Parse response like "coca cola: 45%"
                if ':' in response:
                    cand, perc_str = response.split(':', 1)
                    cand = cand.strip()
                    perc_str = perc_str.strip().replace('%', '')

                    try:
                        if cand == c and perc_str.replace('.', '', 1).isdigit():
                            percentage = float(perc_str)
                            if 0 <= percentage <= 100:
                                raw_predictions[c] = percentage
                            else:
                                print(f"Out-of-bounds percentage: {percentage}")
                        else:
                            print(f"Candidate mismatch or bad format: '{response}'")
                    except Exception as e:
                        print(f"Failed to parse response: {response} — {e}")
                else:
                    print(f"Invalid format: {response}")


            total_raw = sum(raw_predictions.values())
            if total_raw > 0:
                for c in raw_predictions:
                    normalized = raw_predictions[c] / total_raw
                    totals[c] += normalized
            else:
                skipped_rows += 1
                print("Skipped this person due to 0% total across all candidates.")

        print(f"\nFinished polling. Skipped rows due to invalid predictions: {skipped_rows}")
        return totals


if __name__ == "__main__":
    vm_file = "variable_mapping.csv"
    api_key = input("Enter your Azure OpenAI API key: ")
    endpoint = input("Enter your Azure endpoint URL: ")
    deployment_name = input("Enter your deployment name: ")

    raw = input("Enter all candidate names separated by commas: ")
    candidates = [c.strip() for c in raw.split(",")]

    state_file = "test_table.csv"
    state_name = "Alaska"

    start_time = time.time()
    poller = PoliticalLLM(vm_file, candidates, api_key, endpoint, deployment_name)
    poller.load_state(state_file)
    results = poller.poll(state_name)

    winner = max(results, key=results.get)
    print ("Normalized result:")
    print(f"\nThe winner is {winner} with {results[winner]:.2f} votes.")
    print(f"Finished in {time.time() - start_time:.2f} seconds.")



Processing row 1/50 in state Alaska...
pepsi: pepsi: 48%
coca cola: coca cola: 53%

Processing row 2/50 in state Alaska...
pepsi: pepsi: 48%
coca cola: coca cola: 60%

Processing row 3/50 in state Alaska...
pepsi: pepsi: 52%
coca cola: coca cola: 52%

Processing row 4/50 in state Alaska...
pepsi: pepsi: 45%
coca cola: coca cola: 53%

Processing row 5/50 in state Alaska...
pepsi: pepsi: 47%
coca cola: coca cola: 51%

Processing row 6/50 in state Alaska...
pepsi: pepsi: 45%
coca cola: coca cola: 52%

Processing row 7/50 in state Alaska...
pepsi: pepsi: 45%
coca cola: coca cola: 54%

Processing row 8/50 in state Alaska...
pepsi: pepsi: 45%
coca cola: coca cola: 52%

Processing row 9/50 in state Alaska...
pepsi: pepsi: 45%
coca cola: coca cola: 52%

Processing row 10/50 in state Alaska...
pepsi: pepsi: 47%
coca cola: coca cola: 52%

Processing row 11/50 in state Alaska...
pepsi: pepsi: 45%
coca cola: coca cola: 53%

Processing row 12/50 in state Alaska...
pepsi: pepsi: 48%
coca cola: coca

One-at-a-time approach does not give normalized result, so I normalized them manually.
However, this time, coca cola is the winner.
Two approaches return different results, because All-at-once prompt distribute attention on different options,
whereas One-at-a-time approach focus on each candidate, has more quality but the outputs are less logic (need to normalize).


However, All-at-once (55.97 seconds) is more efficient than One-at-a-time approach(85.34 seconds)

In conclusion, to compare two approaches for predicting election outcomes using LLMs, we tested a one-at-a-time method (querying each candidate separately) and an all-at-once method (asking for all candidate probabilities together). While both methods produced different winners in our Alaska test case, the all-at-once approach proved superior in speed, consistency, and alignment with how LLMs reason over full context. It avoids normalization issues, reduces API calls, and yields more interpretable results, making it the preferred choice for scalable, group-based election prediction.